# 05 — Model evaluation

Generates and displays confusion matrices, ROC, and feature importance.

**Prerequisite:** trained artifacts in `models/trained_models/`.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path("..").resolve()
fig_dir = ROOT / "models" / "trained_models" / "figures"
report_path = ROOT / "models" / "trained_models" / "training_report.json"

subprocess.check_call(["python", str(ROOT / "scripts" / "05_plot_evaluation.py")], cwd=ROOT)
report = json.loads(report_path.read_text())

display(Markdown("## Binary test metrics"))
bin_row = {k: v for k, v in report["binary"]["test"].items() if k != "confusion_matrix"}
display(pd.DataFrame([bin_row]))

display(Markdown("## Multiclass test metrics"))
mc = {k: v for k, v in report["multiclass"]["test"].items() if k != "confusion_matrix"}
display(pd.DataFrame([mc]))

In [ ]:
display(Markdown("## Validation model comparison (binary)"))
rows = []
for name, m in report["binary"]["validation"].items():
    rows.append({
        "model": name,
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "roc_auc": m.get("roc_auc"),
    })
display(pd.DataFrame(rows).sort_values("f1", ascending=False))

In [ ]:
for name, title in [
    ("binary_confusion_matrix.png", "Binary confusion matrix"),
    ("binary_roc.png", "Binary ROC"),
    ("multiclass_confusion_matrix.png", "Multiclass confusion matrix"),
    ("feature_importance.png", "Feature importance"),
]:
    path = fig_dir / name
    display(Markdown(f"### {title}"))
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing", path)

## Takeaways for the report

- Prefer **recall/precision/F1** over accuracy under class imbalance.
- **XGBoost** won both stages on the full filtered CICIDS2017 run.
- Macro-F1 < weighted-F1 because minority families (Bot/WebAttack) remain harder.